# Import Libraries 

In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.svm import LinearSVC

# Load Cleaned Data

In [6]:
df = pd.read_csv("cleaned_resume_data.csv")
df.head()

,Resume_str,Category,cleaned
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr,hr administrator marketing associate hr admini...
1,"HR SPECIALIST, US HR OPERATIONS ...",hr,hr specialist us hr operations summary versati...
2,HR DIRECTOR Summary Over 2...,hr,hr director summary 20 years experience recrui...
3,HR SPECIALIST Summary Dedica...,hr,hr specialist summary dedicated driven dynamic...
4,HR MANAGER Skill Highlights ...,hr,hr manager skill highlights hr skills hr depar...


# Prepare Features and Labels

In [7]:
X = df['cleaned']      # input text
y = df['Category']     # target labels

# check distribution
y.value_counts()

Category
business                  234
information-technology    120
business-development      119
legal                     118
chef                      118
engineering               118
finance                   118
fitness                   117
aviation                  117
healthcare                115
banking                   115
consultant                115
construction              112
public-relations          111
hr                        110
designer                  107
arts                      103
education                 102
apparel                    97
digital-media              96
agriculture                63
Name: count, dtype: int64

# Train-Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # keeps class balance
)

# Build Pipeline (TF-IDF and  Model also)

In [9]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words='english',
        max_features=50000,
        ngram_range=(1,2),
        min_df=2,
        max_df=0.9
    )),
    ("clf", LinearSVC(
        C=1.5,
        class_weight='balanced'
    ))
])

# Hyperparameter Tuning

In [10]:
param_grid = {
    "tfidf__ngram_range": [(1,1), (1,2)],
    "tfidf__max_features": [20000, 30000],
    "tfidf__min_df": [2],
    "tfidf__max_df": [0.9, 0.95],
    "tfidf__stop_words": [None, "english"],
    "tfidf__sublinear_tf": [True, False],
    "clf__C": [0.5, 1, 2],
    "clf__class_weight": [None, "balanced"]
}


# Grid Search (To find good model parameters)

In [11]:
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)

Fitting 3 folds for each of 192 candidates, totalling 576 fits
Best Parameters: {'clf__C': 2, 'clf__class_weight': None, 'tfidf__max_df': 0.9, 'tfidf__max_features': 20000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__stop_words': None, 'tfidf__sublinear_tf': False}


# Evaluate Model

In [14]:
y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.7175257731958763


# Detailed Performance

In [16]:
# classification report
print(classification_report(y_test, y_pred))

                        precision    recall  f1-score   support

           agriculture       0.50      0.38      0.43        13
               apparel       0.75      0.47      0.58        19
                  arts       0.39      0.33      0.36        21
              aviation       0.82      0.78      0.80        23
               banking       0.73      0.70      0.71        23
              business       0.66      0.70      0.68        47
  business-development       0.71      0.92      0.80        24
                  chef       0.72      0.75      0.73        24
          construction       0.79      0.86      0.83        22
            consultant       0.69      0.48      0.56        23
              designer       0.74      0.81      0.77        21
         digital-media       0.62      0.53      0.57        19
             education       0.75      0.90      0.82        20
           engineering       0.73      0.79      0.76        24
               finance       0.95      

# Confusion Matrix

In [17]:
cm = confusion_matrix(y_test, y_pred)

pd.DataFrame(cm, index=best_model.classes_, columns=best_model.classes_)

,agriculture,apparel,arts,aviation,banking,business,business-development,chef,construction,consultant,...,digital-media,education,engineering,finance,fitness,healthcare,hr,information-technology,legal,public-relations
agriculture,5,0,2,0,0,3,0,0,0,0,...,0,1,0,0,0,0,0,1,1,0
apparel,0,9,1,0,3,1,0,0,1,1,...,0,0,1,0,0,0,1,0,0,0
arts,0,0,7,0,0,1,1,3,0,0,...,1,2,0,0,1,2,0,0,0,2
aviation,0,0,0,18,0,1,0,0,0,0,...,0,0,3,0,0,0,0,1,0,0
banking,0,0,1,0,16,0,3,0,0,1,...,1,0,0,0,0,0,0,1,0,0
business,1,0,1,3,1,33,1,1,0,1,...,1,0,0,1,1,1,0,0,0,0
business-development,0,0,0,0,1,0,22,0,0,0,...,1,0,0,0,0,0,0,0,0,0
chef,0,1,1,0,0,0,0,18,0,0,...,0,1,0,0,0,2,0,0,1,0
construction,0,0,0,0,0,1,0,0,19,0,...,0,0,0,0,0,0,0,0,0,1
consultant,1,1,0,0,0,1,1,1,1,11,...,0,0,1,0,0,1,0,3,0,1


# Cross-Validation Score

In [18]:
cv_scores = cross_val_score(best_model, X, y, cv=3)

print("CV Scores:", cv_scores)
print("Average CV Score:", cv_scores.mean())

CV Scores: [0.73176761 0.72648515 0.6509901 ]
Average CV Score: 0.7030809539544807


# Test Custom Input

In [19]:
def predict_resume(text):
    return best_model.predict([text])[0]

# example
sample = "python machine learning pandas numpy data analysis"
predict_resume(sample)

'engineering'